# Prism — Train EfficientNet-B0 — Slot E (pretrained, second)
**Ron — second pretrained method**

ImageNet-pretrained EfficientNet-B0 (2019), fine-tuned on CIFAR-10.

### Colab setup
1. Runtime > Change runtime type > **T4 GPU**.
2. Clone the repo: `!git clone https://github.com/JethroIsHere/prism.git`
   (or upload the project folder and adjust `PROJECT_ROOT` below).
3. Run the cells top to bottom. GPU fine-tuning takes ~30 minutes.


In [ ]:
import json, os, sys, time
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Locate the project root: this notebook lives in ron-efficientnet/
_cwd = os.getcwd()
if os.path.isdir(os.path.join(_cwd, 'ron-efficientnet')):
    PROJECT_ROOT = _cwd
elif os.path.basename(_cwd) == 'ron-efficientnet':
    PROJECT_ROOT = os.path.dirname(_cwd)
else:
    PROJECT_ROOT = "/content/prism" if os.path.isdir("/content/prism") else "."
print("PROJECT_ROOT:", PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, "ron-efficientnet"))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "shared"))

from efficientnet_finetune import (build_model, count_parameters, fit,
                                   evaluate, plot_curves,
                                   confusion_matrix,
                                   accuracy_and_macro_f1,
                                   plot_confusion_matrix)
from data_utils import load_canonical_splits, CLASS_NAMES, SEED

RESULTS = os.path.join(PROJECT_ROOT, "ron-efficientnet", "results")
os.makedirs(RESULTS, exist_ok=True)

# ImageNet statistics -- the pretrained weights expect this normalization.
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
EPOCHS = 12
BATCH_SIZE = 64
LR = 1e-4
# 224 is the ImageNet recipe (Colab GPU). On CPU-only machines set PRISM_INPUT_SIZE=32 -- 224px costs ~130h here (measured).INPUT_SIZE = int(os.environ.get("PRISM_INPUT_SIZE", "224")),device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
torch.manual_seed(SEED); np.random.seed(SEED)


In [ ]:
# CIFAR-10 download (one-time, ~170MB) — skipped if already present.
import tarfile, urllib.request
_DATA = os.path.join(PROJECT_ROOT, "data")
_READY = os.path.join(_DATA, "cifar-10-batches-py", "test_batch")
os.makedirs(_DATA, exist_ok=True)
if not os.path.isfile(_READY):
    print("Downloading CIFAR-10 (~170MB)...")
    _tgz = os.path.join(_DATA, "cifar-10-python.tar.gz")
    urllib.request.urlretrieve(
        "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz", _tgz)
    print("Extracting...")
    with tarfile.open(_tgz, "r:gz") as _tf:
        for _m in _tf.getmembers():
            if _m.isfile():  # manual copy: avoids tar ownership quirks
                _out = os.path.join(_DATA, _m.name)
                os.makedirs(os.path.dirname(_out), exist_ok=True)
                with open(_out, 'wb') as _f:
                    _f.write(_tf.extractfile(_m).read())
    print("CIFAR-10 ready.")
else:
    print("CIFAR-10 already present.")


## 1. Data

The canonical 45k train / 5k val / 10k test splits (stratified,
seed 0) — the same split every Prism method uses. EfficientNet-B0
expects 224×224 inputs with ImageNet normalisation, so images are
resized from 32×32; training adds a random horizontal flip.


In [ ]:
Xtr, ytr, Xva, yva, Xte, yte = load_canonical_splits(
    os.path.join(PROJECT_ROOT, 'data', 'cifar-10-batches-py'))
print(f'train {Xtr.shape}, val {Xva.shape}, test {Xte.shape}')

class CifarDataset(Dataset):
    def __init__(self, images, labels, transform):
        self.images, self.labels = images, labels
        self.transform = transform
    def __len__(self):
        return len(self.images)
    def __getitem__(self, i):
        return self.transform(self.images[i]), int(self.labels[i])

train_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(INPUT_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
train_loader = DataLoader(CifarDataset(Xtr, ytr, train_tf),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2,
                          pin_memory=(device == 'cuda'))
val_loader = DataLoader(CifarDataset(Xva, yva, eval_tf),
                        batch_size=256, num_workers=(2 if device == 'cuda' else 0))
test_loader = DataLoader(CifarDataset(Xte, yte, eval_tf),
                         batch_size=256, num_workers=2)
log = []


## 2. Model

EfficientNet-B0 (2019) with ImageNet weights; only the final Linear
layer is replaced (1280 → 10). Compound scaling + MBConv blocks with
squeeze-and-excitation — 5.3M parameters, the smallest of the family.


In [ ]:
print("Loading ImageNet-pretrained EfficientNet-B0...")
model = build_model(num_classes=10, pretrained=True).to(device)
n_params = count_parameters(model)
print(f"EfficientNet-B0: {n_params:,} trainable parameters ({n_params/1e6:.2f}M)")


## 3. Fine-tune

Adam at a small learning rate (1e-4) with cosine decay, 12 epochs.
The best-validation checkpoint is kept — that is the model that gets
evaluated and saved.


In [ ]:
print(f"Fine-tuning ({EPOCHS} epochs, Adam lr={LR})...")
t0 = time.perf_counter()
history = fit(model, train_loader, val_loader, device,
              epochs=EPOCHS, lr=LR, verbose=True)
train_time_min = (time.perf_counter() - t0) / 60
print(f"Fine-tuning took {train_time_min:.1f} min")
log.append(f"training_time_minutes: {train_time_min:.1f}")
torch.save(model.state_dict(), os.path.join(RESULTS, 'model.pt'))
plot_curves(history, os.path.join(RESULTS, 'training_curves.png'))
print("Saved training_curves.png")


## 4. Evaluation (run once on the test set)


In [ ]:
criterion = torch.nn.CrossEntropyLoss()
model.eval()
t0 = time.perf_counter()
with torch.no_grad():
    for images, _ in test_loader:
        _ = model(images.to(device))
        if device == 'cuda':
            torch.cuda.synchronize()
ms_per_image = (time.perf_counter() - t0) / len(Xte) * 1000
print(f"{ms_per_image:.2f} ms/image ({device})")
_, _, y_pred, y_true = evaluate(model, test_loader, criterion, device)

cm = confusion_matrix(y_true, y_pred)
acc, macro_f1, per_class_f1 = accuracy_and_macro_f1(cm)
print(f"Top-1 accuracy: {acc*100:.2f}%")
print(f"Macro F1: {macro_f1:.4f}")
log.append(f"top1_accuracy_pct: {acc*100:.2f}")
log.append(f"macro_f1: {macro_f1:.4f}")

metrics = {
    "method": "EfficientNet-B0 (ImageNet pretrained, fine-tuned)",
    "dataset": "CIFAR-10",
    "n_train": int(len(Xtr)),
    "n_val": int(len(Xva)),
    "n_test": int(len(Xte)),
    "top1_accuracy_pct": round(acc*100, 2),
    "macro_f1": round(macro_f1, 4),
    "per_class_f1": {n: round(float(v), 4) for n, v in zip(CLASS_NAMES, per_class_f1)},
    "runtime_ms_per_image": round(ms_per_image, 2),
    "runtime_device": device,
    "parameter_count_millions": round(n_params/1e6, 2),
    "training_time_minutes": round(train_time_min, 1),
    "hyperparams": {"epochs": EPOCHS, "batch_size": BATCH_SIZE,
                  "lr": LR, "optimizer": "Adam",
                  "weight_decay": 1e-4, "scheduler": "cosine",
                  "input_size": INPUT_SIZE,                  "preprocessing": f"resize{INPUT_SIZE}+imagenet_norm",
                  "augmentation": "flip"},
}
with open(os.path.join(RESULTS, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
np.save(os.path.join(RESULTS, "confusion_matrix.npy"), cm)
plot_confusion_matrix(cm, CLASS_NAMES,
                      f"EfficientNet-B0 confusion matrix (acc {acc*100:.2f}%)",
                      os.path.join(RESULTS, "confusion_matrix.png"))
with open(os.path.join(RESULTS, "training_log.txt"), "w") as f:
    f.write("\n".join(log) + "\n")
print("Saved: metrics.json, confusion_matrix.png/.npy, model.pt, training_log.txt")
